In [ ]:
!pip install img2dataset datasets

In [ ]:
import os
from datasets import load_dataset
import csv

BASE_DIR = "/home/rahul/shyam/aditya/training/cc3m"
SHARDS_DIR = f"{BASE_DIR}/shards"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(SHARDS_DIR, exist_ok=True)

print("Loading CC3M metadata...")
cc3m_meta = load_dataset(
    "google-research-datasets/conceptual_captions",
    split="train",
    trust_remote_code=True
)
print(f"Total samples: {len(cc3m_meta)}")
print(f"Sample: {cc3m_meta[0]}")

In [ ]:
url_file = f"{BASE_DIR}/cc3m_urls.tsv"

print(f"Writing URLs to {url_file}...")
with open(url_file, 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['url', 'caption'])
    for sample in cc3m_meta:
        writer.writerow([sample['image_url'], sample['caption']])
print(f"Done: {len(cc3m_meta)} rows written")

In [ ]:
!img2dataset \
    --url_list "/home/rahul/shyam/aditya/training/cc3m/cc3m_urls.tsv" \
    --input_format tsv \
    --url_col url \
    --caption_col caption \
    --output_format webdataset \
    --output_folder "/home/rahul/shyam/aditya/training/cc3m/shards" \
    --processes_count 8 \
    --thread_count 64 \
    --image_size 336 \
    --resize_mode keep_ratio \
    --min_image_size 100 \
    --max_aspect_ratio 3.0 \
    --number_sample_per_shard 5000 \
    --retries 2 \
    --enable_wandb False

In [ ]:
import glob, os

shards = sorted(glob.glob("/home/rahul/shyam/aditya/training/cc3m/shards/*.tar"))
total_size = sum(os.path.getsize(f) for f in shards) / 1e9

print(f"Shards downloaded : {len(shards)}")
print(f"Disk used         : {total_size:.1f} GB")
print(f"Estimated images  : ~{len(shards) * 5000 * 0.55:,.0f}")

In [ ]:
from transformers import AutoModel, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
import torch

model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")

lora_config = LoraConfig(
    r=8, lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.FEATURE_EXTRACTION,
)
peft_model = get_peft_model(model, lora_config)

# Confirm embed_tokens path
from model import get_embed_tokens
embed_fn = get_embed_tokens(peft_model)
print(type(embed_fn))  # should be nn.Embedding

# Confirm forward works
ids = tokenizer(["test sentence"], return_tensors="pt")["input_ids"]
embeds = embed_fn(ids)
print(embeds.shape)  # [1, seq_len, 1024]